In [1]:
import sys
import time
import json
import numpy as np
import torch
from torch.nn import Parameter
from pathlib import Path

gems_tco_path = "/Users/joonwonlee/Documents/GEMS_TCO-1/src"
sys.path.append(gems_tco_path)

from GEMS_TCO import configuration as config
from GEMS_TCO import debiased_whittle as debiased_whittle
from GEMS_TCO import debiased_whittle_lat1d as debiased_whittle
from GEMS_TCO.data_loader import load_data_dynamic_processed

Load daily data applying max-min ordering

In [2]:
lat_lon_resolution = [1, 1]
mm_cond_number = 8
years = ['2022']
month_range = [7]

output_path = Path(config.mac_estimates_day_path)
data_load_instance = load_data_dynamic_processed(config.mac_data_load_path)

lat_range_input = [-3, 2]
lon_range_input = [121, 131]

df_map, ord_mm, nns_map, monthly_mean = data_load_instance.load_maxmin_ordered_data_bymonthyear(
    lat_lon_resolution=lat_lon_resolution,
    mm_cond_number=mm_cond_number,
    years_=years,
    months_=month_range,
    lat_range=lat_range_input,
    lon_range=lon_range_input,
    is_whittle=True
)

--- Global Monthly Mean for 2022-7: 250.6500 ---


In [4]:
daily_aggregated_tensors_dw = []
daily_hourly_maps_dw = []

for day_index in range(31):
    hour_indices = [day_index * 8, (day_index + 1) * 8]

    day_hourly_map, day_aggregated_tensor = data_load_instance.load_working_data(
        df_map,
        monthly_mean,
        hour_indices,
        ord_mm=None,
        dtype=torch.float64,
        keep_ori=False
    )
    daily_aggregated_tensors_dw.append(day_aggregated_tensor)
    daily_hourly_maps_dw.append(day_hourly_map)

print(f"Loaded {len(daily_hourly_maps_dw)} days. Sample shape: {daily_aggregated_tensors_dw[0].shape}")

Loaded 31 days. Sample shape: torch.Size([145008, 11])


whittle

In [5]:
DEVICE_DW = torch.device("cpu")

dwl = debiased_whittle.debiased_whittle_likelihood()
TAPERING_FUNC = dwl.cgn_hamming
DWL_MAX_STEPS = 15
LAT_COL, LON_COL, VAL_COL, TIME_COL = 0, 1, 2, 3

days_list = [0]  # 0-based index

for day_idx in days_list:
    print(f'\n{"="*50}')
    print(f'--- DW: Day {day_idx+1} (2024-07-{day_idx+1}) ---')
    print(f'{"="*50}')

    start_time = time.time()

    try:
        daily_hourly_map_dw = daily_hourly_maps_dw[day_idx]
        daily_aggregated_tensor_dw = daily_aggregated_tensors_dw[day_idx].to(DEVICE_DW)

        if daily_aggregated_tensor_dw.shape[0] == 0:
            print(f"Skipping Day {day_idx+1}: No data.")
            continue

        init_sigmasq    = 13.059
        init_range_lat  = 0.154
        init_range_lon  = 0.195
        init_range_time = 0.7
        init_advec_lat  = 0.0218
        init_advec_lon  = -0.1689
        init_nugget     = 0.247

        raw_init_floats = [
            init_sigmasq, init_range_lat, init_range_lon, init_range_time,
            init_advec_lat, init_advec_lon, init_nugget
        ]

        init_phi2 = 1.0 / init_range_lon
        init_phi1 = init_sigmasq * init_phi2
        init_phi3 = (init_range_lon / init_range_lat) ** 2
        init_phi4 = (init_range_lon / init_range_time) ** 2

        initial_vals = [
            np.log(init_phi1), np.log(init_phi2), np.log(init_phi3),
            np.log(init_phi4), init_advec_lat, init_advec_lon, np.log(init_nugget)
        ]

        params_list = [
            Parameter(torch.tensor([val], dtype=torch.float64, device=DEVICE_DW))
            for val in initial_vals
        ]

        # ── Preprocess ───────────────────────────────────────────────────────
        db = debiased_whittle.debiased_whittle_preprocess(
            daily_aggregated_tensors_dw, daily_hourly_maps_dw,
            day_idx=day_idx, params_list=raw_init_floats,
            lat_range=[-3, 2], lon_range=[121.0, 131.0]
        )

        cur_df = db.generate_spatially_filtered_days(-3, 2, 121, 131).to(DEVICE_DW)

        if cur_df.numel() == 0 or cur_df.shape[1] <= max(LAT_COL, LON_COL, VAL_COL, TIME_COL):
            print(f"Error: Data for Day {day_idx+1} is empty or invalid.")
            continue

        unique_times = torch.unique(cur_df[:, TIME_COL])
        time_slices_list = [cur_df[cur_df[:, TIME_COL] == t_val] for t_val in unique_times]

        # ── Pre-compute J-vector, periodogram, taper autocorr ────────────────
        print("Pre-computing J-vector (Hamming taper)...")
        J_vec, n1, n2, p_time, taper_grid, obs_masks = dwl.generate_Jvector_tapered_mv(
            time_slices_list, tapering_func=TAPERING_FUNC,
            lat_col=LAT_COL, lon_col=LON_COL, val_col=VAL_COL, device=DEVICE_DW
        )

        if J_vec is None or J_vec.numel() == 0 or n1 == 0 or n2 == 0:
            print(f"Error: J-vector generation failed for Day {day_idx+1}.")
            continue

        I_sample = dwl.calculate_sample_periodogram_vectorized(J_vec)
        taper_autocorr_grid = dwl.calculate_taper_autocorrelation_multivariate(taper_grid, obs_masks, n1, n2, DEVICE_DW)
        del obs_masks
        print(f"Grid: {n1}x{n2}, {p_time} time points.")

        # ── Optimize ──────────────────────────────────────────────────────────
        optimizer = torch.optim.LBFGS(
            params_list, lr=1.0, max_iter=DWL_MAX_STEPS, history_size=100,
            line_search_fn="strong_wolfe", tolerance_grad=1e-5
        )

        nat_str, phi_str, raw_str, loss, steps_run = dwl.run_lbfgs_tapered(
            params_list=params_list, optimizer=optimizer, I_sample=I_sample,
            n1=n1, n2=n2, p_time=p_time, taper_autocorr_grid=taper_autocorr_grid,
            max_steps=DWL_MAX_STEPS, device=DEVICE_DW
        )

        epoch_time = time.time() - start_time
        print(f"\nBest Loss: {loss} (after {steps_run} steps, {epoch_time:.2f}s)")
        print(f"Natural Scale: {nat_str}")
        print(f"Phi Scale    : {phi_str}")
        print(f"Raw Log Scale: {raw_str}")

    except Exception as e:
        import traceback
        print(f"Day {day_idx+1} Failed: {e}")
        traceback.print_exc()
        continue


--- DW: Day 1 (2024-07-1) ---
Pre-computing J-vector (Hamming taper)...
Grid: 113x159, 8 time points.
--- Step 1/15 ---
 Loss: -8.966479 | Max Grad: 4.150473e-01
  Params (Raw Log): log_phi1: 3.6605, log_phi2: 2.2458, log_phi3: 0.1460, log_phi4: -4.9731, advec_lat: -0.0253, advec_lon: 0.0623, log_nugget: 0.6283
--- Step 2/15 ---
 Loss: -9.544101 | Max Grad: 4.005045e-03
  Params (Raw Log): log_phi1: 2.9428, log_phi2: 1.5611, log_phi3: 0.6672, log_phi4: -3.8170, advec_lat: -0.0137, advec_lon: 0.0653, log_nugget: 0.9147
--- Step 3/15 ---
 Loss: -9.621655 | Max Grad: 2.627868e-05
  Params (Raw Log): log_phi1: 2.9426, log_phi2: 1.5537, log_phi3: 0.6656, log_phi4: -3.8183, advec_lat: -0.0139, advec_lon: 0.0660, log_nugget: 0.9154
--- Step 4/15 ---
 Loss: -9.621664 | Max Grad: 2.627868e-05
  Params (Raw Log): log_phi1: 2.9426, log_phi2: 1.5537, log_phi3: 0.6656, log_phi4: -3.8183, advec_lat: -0.0139, advec_lon: 0.0660, log_nugget: 0.9154
--- Step 5/15 ---
 Loss: -9.621664 | Max Grad: 2.6278

==================================================
--- DW: Day 2 (2024-07-2) ---
==================================================
Pre-computing J-vector (Hamming taper)...
Grid: 113x158, 8 time points.
--- Step 1/15 ---
 Loss: -2.395891 | Max Grad: 1.043809e-03
  Params (Raw Log): log_phi1: 2.9467, log_phi2: 1.3769, log_phi3: 0.4596, log_phi4: -2.4607, advec_lat: 0.0164, advec_lon: -0.4929, log_nugget: -0.2484
--- Step 2/15 ---
 Loss: -4.069132 | Max Grad: 2.794097e-06
  Params (Raw Log): log_phi1: 2.9452, log_phi2: 1.3740, log_phi3: 0.4604, log_phi4: -2.4203, advec_lat: 0.0166, advec_lon: -0.4925, log_nugget: -0.2469
--- Step 3/15 ---
 Loss: -4.069153 | Max Grad: 2.794097e-06
  Params (Raw Log): log_phi1: 2.9452, log_phi2: 1.3740, log_phi3: 0.4604, log_phi4: -2.4203, advec_lat: 0.0166, advec_lon: -0.4925, log_nugget: -0.2469
--- Step 4/15 ---
 Loss: -4.069153 | Max Grad: 2.794097e-06
  Params (Raw Log): log_phi1: 2.9452, log_phi2: 1.3740, log_phi3: 0.4604, log_phi4: -2.4203, advec_lat: 0.0166, advec_lon: -0.4925, log_nugget: -0.2469

--- Converged on gradient norm (max|grad| < 1e-05) at step 4 ---

--- Training Complete ---

FINAL BEST STATE ACHIEVED (during training):
Best Loss: -4.069

Best Loss: -4.069 (after 4 steps, 13.35s)
Natural Scale: sigmasq: 4.8123, range_lat: 0.2010, range_lon: 0.2531, range_time: 0.8489, advec_lat: 0.0166, advec_lon: -0.4925, nugget: 0.7812
Phi Scale    : phi1: 19.0140, phi2: 3.9512, phi3: 1.5847, phi4: 0.0889, advec_lat: 0.0166, advec_lon: -0.4925, nugget: 0.7812
Raw Log Scale: log_phi1: 2.9452, log_phi2: 1.3740, log_phi3: 0.4604, log_phi4: -2.4203, advec_lat: 0.0166, advec_lon: -0.4925, log_nugget: -0.2469

========================= Overall Result from Run ========================= =========================
Best Run Loss: -1.751 (after 4 steps)
Final Parameters (Natural Scale): sigmasq: 4.8532, range_lat: 0.1364, range_lon: 0.1507, range_time: 0.9679, advec_lat: -0.0059, advec_lon: -0.2527, nugget: 0.9001
Final Parameters (Phi Scale)    : phi1: 32.2102, phi2: 6.6369, phi3: 1.2208, phi4: 0.0242, advec_lat: -0.0059, advec_lon: -0.2527, nugget: 0.9001
Final Parameters (Raw Log Scale): log_phi1: 3.4723, log_phi2: 1.8926, log_phi3: 0.1995, log_phi4: -3.7200, advec_lat: -0.0059, advec_lon: -0.2527, log_nugget: -0.1053



========================= Overall Result from Run ========================= =========================
Best Run Loss: -1.775 (after 4 steps)
Final Parameters (Natural Scale): sigmasq: 4.7654, range_lat: 0.1343, range_lon: 0.1482, range_time: 0.9408, advec_lat: -0.0055, advec_lon: -0.2387, nugget: 0.8993
Final Parameters (Phi Scale)    : phi1: 32.1604, phi2: 6.7487, phi3: 1.2181, phi4: 0.0248, advec_lat: -0.0055, advec_lon: -0.2387, nugget: 0.8993
Final Parameters (Raw Log Scale): log_phi1: 3.4707, log_phi2: 1.9094, log_phi3: 0.1973, log_phi4: -3.6967, advec_lat: -0.0055, advec_lon: -0.2387, log_nugget: -0.1061

Total execution time: 18.07 seconds